In [0]:
%pip install PyMuPDF azure-ai-formrecognizer

  Using cached pymupdf-1.26.7-cp310-abi3-manylinux_2_28_x86_64.whl.metadata (3.4 kB)
Using cached pymupdf-1.26.7-cp310-abi3-manylinux_2_28_x86_64.whl (24.1 MB)
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
import fitz
from azure.ai.formrecognizer import DocumentAnalysisClient
from azure.core.credentials import AzureKeyCredential

In [0]:
# credentials for Azure form recognizer
# must add in endpoint and key for this to run
endpoint = None # insert endpoint
key = None # insert key

print("Initializing Azure Form Reciognizer Client")

client = DocumentAnalysisClient(endpoint=endpoint, credential=AzureKeyCredential(key))

Initializing Azure Form Reciognizer Client


---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-5729382867801708>, line 7
      3 key = None # insert key
      5 print("Initializing Azure Form Reciognizer Client")
----> 7 client = DocumentAnalysisClient(endpoint=endpoint, credential=AzureKeyCredential(key))

NameError: name 'DocumentAnalysisClient' is not defined

In [0]:
# funcion to run Azure ocr (thanks Arsh)

def ocr_pdf(binary, path):
    print(f"[OCR] Extracting Text")
    try:
        poller = client.begin_analyze_document("prebuilt-read", document=binary)
        result = poller.result()
        text="\n".join(
            line.content
            for page in result.pages
            for line in page.lines
        )
        print(f"[OCR] Success ({len(text)}chars)")
        return text
    except Exception as e:
        print(f"[OCR ERROR] {path} -> {e}")
        return ""

In [0]:
# reading files from DBFS
pdf_df=spark.read.format("binaryFile").option("recursiveFileLookup","true").load("dbfs:/FileStore/easyocr_vs_azure_sampled_pdfs").select("path", "content")

In [0]:
# data structure schema for ocr output

from pyspark.sql.types import StructType, StructField, StringType, BooleanType
from pyspark.sql.functions import col, length

schema=StructType([
    StructField("path", StringType(), True),
    StructField("final_text", StringType(), True)
])


In [0]:
# running Azure OCR

rows=pdf_df.toLocalIterator()

for idx, row in enumerate(rows, start=1):
    path=row["path"]
    binary=row["content"]

    final_text=ocr_pdf(binary, path)

    result_df=spark.createDataFrame([(path, final_text)], schema)
    result_df.write.format("delta").mode("append").saveAsTable("azure_ocr_exp")

    print("[SUCCESS] File processed")

print("JOB FINISHED")

[OCR] Extracting Text
[OCR] Success (4955chars)
[SUCCESS] File processed
[OCR] Extracting Text
[OCR] Success (1777chars)
[SUCCESS] File processed
[OCR] Extracting Text
[OCR] Success (2720chars)
[SUCCESS] File processed
[OCR] Extracting Text
[OCR] Success (1744chars)
[SUCCESS] File processed
[OCR] Extracting Text
[OCR] Success (2145chars)
[SUCCESS] File processed
[OCR] Extracting Text
[OCR] Success (2104chars)
[SUCCESS] File processed
[OCR] Extracting Text
[OCR] Success (1809chars)
[SUCCESS] File processed
[OCR] Extracting Text
[OCR] Success (1561chars)
[SUCCESS] File processed
[OCR] Extracting Text
[OCR] Success (2607chars)
[SUCCESS] File processed
[OCR] Extracting Text
[OCR] Success (2607chars)
[SUCCESS] File processed
[OCR] Extracting Text
[OCR] Success (2607chars)
[SUCCESS] File processed
[OCR] Extracting Text
[OCR] Success (1058chars)
[SUCCESS] File processed
[OCR] Extracting Text
[OCR] Success (1995chars)
[SUCCESS] File processed
[OCR] Extracting Text
[OCR] Success (1788chars)
[SU

In [0]:
display(
    spark.table("azure_ocr_exp")
         .select("path", "final_text")
         .orderBy("path")
)

path final_text dbfs:/FileStore/easyocr_vs_azure_sampled_pdfs/0000005183.png Health
Canada
Santé
Canada
Health Products
and Food Branch
Direction générale des produits
de santé et des aliments
NOTICE OF COMPLIANCE
AVIS DE CONFORMITÉ
AVR
APR
1 8 2006
Sponsor/Manufacturer:/
Promoteur/Fabricant:
9427-S2753-65
sanofi-aventis Canada Inc.
2150 St. Elzear Blvd. W.
Laval, Qc
H7L 4A8
Submission Number/Numéro de présentation:
105035
Product Name/Nom du produit:
LARGACTIL
Medicinal Ingredient(s)/Ingrédient(s) médicinal(aux):
Chlorpromazine (supplied as chlorpromazine hydrochloride)
Chlorpromazine
Therapeutic Classification/Classification Thérapeutique:
Neuroleptic
Reason for Submission/Raison pour présentation:
Manufacturer name change
Drug Identification Number(s), Route(s), Form, Strength/
Identification Numérique de(s) drogue(s), Voie(s), Forme,
Dosage :
01929992 (ORL, DPS, 40mg/ml) - chlorpromazine hcl
01929968 (ORL, LIQ, 25mg/5ml) - chlorpromazine hcl
01929976 (ORL, LIQ, 100mg/5ml) - chlorpromazine hcl
01930001
(RT, SUP, 100mg) - chlorpromazine
This is to notify you that, pursuant to section C.08.004
of the Food and Drug Regulations, the above new drug
submission complies with the requirements of sections C.08.002
and C.08.005.1 of the Regulations. As a manufacturer, you are
further reminded of your obligations under C.08.002(1)(d),
C.08.007 and C.08.008. These obligations are detailed on the
reverse of this notice.
Ceci est pour vous aviser que, conformément à
l'article C.08.004 du règlement sur les aliments et drogues, la
présentation de drogue nouvelle citée en rubrique est conforme
aux exigences des articles C.08.002 et C.08.005.1 des mêmes
règlements. Nous vous rappelons vos obligations en tant que
fabriquant aux termes de C.08.002(1)(d), C.08.007 et C.08.008.
Ces obligations sont expliquées au verso du présent avis.
ORIGINAL SIGNED BY
ORIGINAL SIGNÉ PAR
OMER BOUDREAU
Omer Boudreau
Director General / Directeur général
Therapeutic Products Directorate / Direction des produits thérapeutiques
Enclosures:
Drug Notification Form(s)
Prescribing Information
Pièces jointes:
Formule(s) de déclaration de(s) drogue(s)
Renseignements thérapeutiques
Canada® dbfs:/FileStore/easyocr_vs_azure_sampled_pdfs/0000005183.png Health
Canada
Santé
Canada
Health Products
and Food Branch
Direction générale des produits
de santé et des aliments
NOTICE OF COMPLIANCE
AVIS DE CONFORMITÉ
AVR
APR
1 8 2006
Sponsor/Manufacturer:/
Promoteur/Fabricant:
9427-S2753-65
sanofi-aventis Canada Inc.
2150 St. Elzear Blvd. W.
Laval, Qc
H7L 4A8
Submission Number/Numéro de présentation:
105035
Product Name/Nom du produit:
LARGACTIL
Medicinal Ingredient(s)/Ingrédient(s) médicinal(aux):
Chlorpromazine (supplied as chlorpromazine hydrochloride)
Chlorpromazine
Therapeutic Classification/Classification Thérapeutique:
Neuroleptic
Reason for Submission/Raison pour présentation:
Manufacturer name change
Drug Identification Number(s), Route(s), Form, Strength/
Identification Numérique de(s) drogue(s), Voie(s), Forme,
Dosage :
01929992 (ORL, DPS, 40mg/ml) - chlorpromazine hcl
01929968 (ORL, LIQ, 25mg/5ml) - chlorpromazine hcl
01929976 (ORL, LIQ, 100mg/5ml) - chlorpromazine hcl
01930001
(RT, SUP, 100mg) - chlorpromazine
This is to notify you that, pursuant to section C.08.004
of the Food and Drug Regulations, the above new drug
submission complies with the requirements of sections C.08.002
and C.08.005.1 of the Regulations. As a manufacturer, you are
further reminded of your obligations under C.08.002(1)(d),
C.08.007 and C.08.008. These obligations are detailed on the
reverse of this notice.
Ceci est pour vous aviser que, conformément à
l'article C.08.004 du règlement sur les aliments et drogues, la
présentation de drogue nouvelle citée en rubrique est conforme
aux exigences des articles C.08.002 et C.08.005.1 des mêmes
règlements. Nous vous rappelons vos obligations en tant que
fabriquant aux termes de C.08.002(1)(d), C.08.007 et C.08.008.
Ces obligations sont expliquées au verso du 